# E3 · Límites superiores (Ṁ)

**Spec:** [`docs/spec_E3_codex_upper_limits.md`](../docs/spec_E3_codex_upper_limits.md)  |  **Bloque:** E · Resultado  |  **Run de este set:** `ROXs12b_realigned`

Límite superior de la tasa de acreción Ṁ a partir de la no-detección (Gumbel 99%).

| | |
|---|---|
| **Entrada** | E1 + throughput (E4) + config físico |
| **Salida (QC/productos)** | `stages/stage_h03_qc.json`, `tables/halpha_upper_limits.csv` |
| **Consume aguas abajo** | F1, comparación con G3 |


## Qué hace E3 y cómo sale la cifra

E3 convierte la **no-detección** en un **límite superior de la tasa de acreción Ṁ**. Toma el umbral de flujo **Gumbel 99%** de los máximos de ruido de los controles (matched filter, `f_stat_99`) y lo pasa por la cadena física:

```
f_stat_99  ÷throughput→  f_obs  ×deredden→  f_dered  ×4πd²→  L_Hα  →Alcalá→  Ṁ
```

**Ṁ (99%, canónico psffit) = 8.19e-13 M☉/yr** para este objeto. Insumos físicos (todos citados en el QC): d=138.6 pc, A_V=1.8 (Rizzuto et al. 2015 (moderate-resolution optical spectroscopy), via Bowler et al. 2017 (AJ 154,165)), CCM89 R_V=3.1, relación L_acc–L_Hα de Alcala et al. 2017 Halpha (scatter 0.3 dex), masa 0.0167 M☉, radio 0.135 R☉.

**Dependencia del método = el throughput:** menor throughput → señal peor recuperada → límite **peor** (más alto). El canónico citable es `psffit` (throughput 0.67) por robustez, no por dar el número más bajo: un método puede dar un límite más ajustado y ser menos fiable si su ruido está subestimado en ese borde. **Scatter inter-método 23.1%** (la celda de evidencia y el Plot 1 dan el valor de cada método).

**Nota de definición** ([`docs/mdot_limit_definition_note.md`](../docs/mdot_limit_definition_note.md)): E3 usa Gumbel 99% **sin** el factor R_in 1.25; G3 usa 5σ **con** R_in → G3 da 1.31e-12 (`n/d` = G3 no ha calculado acreción para este objeto; ver su notebook). Misma cadena física; la diferencia es **definicional** (ninguna declarada canónica aún). Knob clave `h03_flux_unit_cgs=1e-20` (unidad nativa scipost) — sin él L/Ṁ salían ~10²⁰ altos.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # el run de este objeto
cd MUSE-accretion-pipeline                    # raíz del repo
bash scripts/stage_h03_limits.sh --run-id $RUN
```

Ligero–moderado.

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
try:
    import matplotlib as mpl
    mpl.rcParams['figure.dpi'] = 120     # retina dobla esto sin agrandar
    mpl.rcParams['savefig.dpi'] = 200
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h03_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'bash scripts/stage_h03_limits.sh --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h03_qc.json', RUN_ID)
nb.show(qc, keys=['canonical_method', 'mdot', 'throughput', 'intermethod_scatter_pct', 'definition'], title='E3')


## Resultados que llevaron a la conclusión

Límite por método, insumos físicos y la nota de definición del `stage_h03_qc.json`.


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E3', 'stages/stage_h03_qc.json'):
        q = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
        lim = {L['method']: L for L in q['limits']}
        pin = q['physical_inputs']
        print('canónico:', q['canonical_method'], f"| scatter inter-método {q['intermethod_scatter_pct']:.0f}%")
        print(f"\nṀ canónico (psffit) = {lim['psffit']['mdot']:.2e} M☉/yr (throughput {lim['psffit']['throughput']:.2f})")
        print('\nṀ por método:')
        for m in ['optimal_psfsub','psffit','aperture','optimal_ls']:
            print(f"   {m:15s} Ṁ={lim[m]['mdot']:.2e}  throughput={lim[m]['throughput']:.2f}")
        print(f"\nfísica: d={pin['distance_pc']}pc, A_V={pin['av']}, A_Hα={pin['a_halpha_over_av']*pin['av']:.2f}, "
              f"masa={pin['companion_mass_msun']:.4f} M☉, radio={pin['companion_radius_rsun']} R☉")
        print(f"relación: {pin['lacc_lha_relation']} (scatter {pin['relation_scatter_dex']} dex)")
        print('\ndefinición: E3 usa Gumbel 99% SIN el factor R_in 1.25 '
              '(G3 usa 5 sigma CON el factor) -> docs/mdot_limit_definition_note.md')
        alt = lim[q['canonical_method']].get('mdot_aoyama21')
        if alt:
            print(f"\nrelación dual (R1): Alcala+17 {lim[q['canonical_method']]['mdot']:.2e}"
                  f"  |  Aoyama+21 {alt:.2e} M☉/yr")
        else:
            print('\n(sin límite Aoyama+21 en este QC: relación alternativa no configurada)')


## Plot 1 — Ṁ por método (el throughput manda)

El límite de Ṁ por método (escala log); en verde el canónico (`psffit` = 8.19e-13 M☉/yr). Cada barra lleva su throughput: los métodos que recuperan peor la señal dan límites **más altos**. Scatter inter-método 23.1%.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
    lim = {L['method']: L for L in q['limits']}
    canon = q['canonical_method']
    methods = ['optimal_psfsub', 'psffit', 'aperture', 'optimal_ls']
    md = [lim[m]['mdot'] for m in methods]; th = [lim[m]['throughput'] for m in methods]
    cols = ['tab:green' if m == canon else '0.6' for m in methods]
    fig, ax = plt.subplots(figsize=(8.5, 4.2))
    ax.bar(range(len(methods)), md, color=cols)
    for i, (m, t) in enumerate(zip(md, th)):
        ax.text(i, m * 1.05, f'{m:.1e}\nT={t:.2f}', ha='center', fontsize=8)
    ax.set_yscale('log'); ax.set_xticks(range(len(methods))); ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylabel('Ṁ límite superior [M☉/yr]')
    ax.set_title(f"E3 · Ṁ por método (canónico {canon}={lim[canon]['mdot']:.1e}; "
                 f"scatter {q['intermethod_scatter_pct']:.0f}%)")
    ax.axhline(lim[canon]['mdot'], color='tab:green', ls='--', lw=1); fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'e3_limits'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'mdot_by_method.png', dpi=110); print('figura ->', outdir / 'mdot_by_method.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Plot 2 — la cadena física: de la no-detección a Ṁ

Los 5 pasos para el método canónico (`psffit`): del umbral de flujo Gumbel 99% (`f_stat_99`), dividir por el throughput, deredden (A_V=1.8), convertir a luminosidad (4πd², d=138.6 pc), y aplicar Alcala et al. 2017 Halpha → **Ṁ = 8.19e-13 M☉/yr**. Cada caja muestra el valor real del QC de este objeto.


In [ ]:
try:
    import matplotlib.pyplot as plt
    q = nb.load_qc('stages/stage_h03_qc.json', RUN_ID)
    canon = q['canonical_method']
    p = {L['method']: L for L in q['limits']}[canon]; pin = q['physical_inputs']
    deredden = p['f_lim_dereddened'] / p['f_lim_observed']
    steps = [('Gumbel 99%\nf_stat_99', p['f_stat_99'], 'erg/s/cm²'),
             (f"÷ throughput\n{p['throughput']:.2f}", p['f_lim_observed'], 'f_obs'),
             (f"× deredden\n(A_Hα={pin['a_halpha_over_av']*pin['av']:.2f}, {deredden:.1f}×)", p['f_lim_dereddened'], 'f_dered'),
             (f"× 4πd²\n(d={pin['distance_pc']}pc)", p['l_halpha'], 'L_Hα [erg/s]'),
             ('Alcalá 2017\n→ L_acc → Ṁ', p['mdot'], 'Ṁ [M☉/yr]')]
    fig, ax = plt.subplots(figsize=(11, 3.4))
    for i, (lab, val, unit) in enumerate(steps):
        ax.text(i, 0.6, lab, ha='center', va='center', fontsize=8, bbox=dict(boxstyle='round', fc='0.92'))
        ax.text(i, 0.25, f'{val:.2e}\n{unit}', ha='center', va='center', fontsize=8, color='tab:blue')
        if i < len(steps) - 1:
            ax.annotate('', (i + 0.65, 0.6), (i + 0.35, 0.6), arrowprops=dict(arrowstyle='->'))
    ax.set_xlim(-0.5, len(steps) - 0.5); ax.set_ylim(0, 1); ax.axis('off')
    ax.set_title(f'E3 · cómo la no-detección se vuelve Ṁ (método canónico {canon})')
    fig.tight_layout()
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'e3_limits'; outdir.mkdir(parents=True, exist_ok=True)
    fig.savefig(outdir / 'physical_chain.png', dpi=110); print('figura ->', outdir / 'physical_chain.png'); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- **Ṁ(99%) = 8.19e-13 M☉/yr** (canónico psffit; Gumbel 99%, L_Hα deredden, throughput 0.67, d=138.6pc, A_V=1.8, Alcala et al. 2017 Halpha). · [`docs/mdot_limit_definition_note.md`](../docs/mdot_limit_definition_note.md)
- Dependencia del método = throughput; **`psffit` canónico citable** por robustez, no por dar el límite más bajo; scatter inter-método 23.1%.
- Difiere de G3 (1.31e-12) solo por DEFINICIÓN (Gumbel99 sin R_in vs 5σ con R_in); cadena física idéntica. Ninguna elegida canónica aún.
- Knob `h03_flux_unit_cgs=1e-20` (unidad nativa scipost) — sin él L/Ṁ salían ~10²⁰ altos.


## Conclusión (registrada)

**E3: Ṁ (99%, canónico psffit) = 8.19e-13 M☉/yr** para este objeto (todo resuelto de su `stage_h03_qc.json`).

- **Cadena:** Gumbel 99% → ÷throughput (0.67) → deredden (A_V=1.8) → ×4πd² (138.6 pc) → Alcala et al. 2017 Halpha.
- **Método:** `psffit` canónico citable por robustez; scatter inter-método 23.1%.
- **vs G3:** 1.31e-12 por definición (5σ + R_in), no por física; ninguna canónica aún.
- **Caveats:** provisional hasta cerrar el A-block del objeto; el FAP 99% está por debajo de la resolución de 33 controles.
- **Resultado científico:** la no-detección de E1 se traduce en el límite superior de Ṁ de arriba (acreción muy baja o ausente).
